# 1. Data Cleaning & Feature Engineering

**Overview & Workflow Sketch**

**Step 1: Standardizing Categorical Labels**
* **Action:** Clean inconsistent entries in the `Item_Fat_Content` column. Map 'LF' and 'low fat' to **'Low Fat'**, and 'reg' to **'Regular'**.
* **Reasoning:** Machine learning models treat 'LF' and 'Low Fat' as entirely different categories. Standardizing them prevents the model from splitting its learning capacity across duplicate categories, thereby avoiding the "dummy variable trap" later during encoding.

**Step 2: Fixing Business Logic Errors**
* **Action:** Identify Non-Consumable items (where `Item_Identifier` starts with 'NC') and change their `Item_Fat_Content` to **'Non-Edible'**.
* **Reasoning:** During EDA, we discovered a logical flaw where items like household goods were labeled as 'Low Fat'. Correcting this separates food items from non-food items, giving the model a more accurate representation of the physical world.

**Step 3: Handling Placeholders (The 0.0 Issue)**
* **Action:** Replace `0.0` values in `Item_Visibility` with `NaN`.
* **Reasoning:** A product physically placed in a store cannot have 0% visibility. The `0.0` is likely a system default for missing data. Treating it as a valid number would severely skew the mean calculation later. We must convert it to `NaN` first to impute it properly.

**Step 4: Imputation for Missing Values**
Instead of using global averages (which introduces high bias), we will use *Grouped Imputation*:
1.  **`Item_Weight` (17.17% missing):** Imputed using the **Mean** weight of the corresponding `Item_Identifier` because the same product code has the exact same physical weight.
2.  **`Item_Visibility` (~6% missing after Step 3):** Imputed using the **Mean** visibility of the corresponding `Item_Identifier` because The same product is likely allocated similar shelf space across different stores.
3.  **`Outlet_Size` (28.28% missing):** Replaced missing values with a new distinct category, **'Unknown'** because since nearly a third of the data is missing, imputing via Mode would introduce severe bias and artificially distort the actual distribution. Treating "missingness" as its own category preserves data integrity and allows the machine learning model to capture any potential hidden patterns associated with unrecorded store sizes.

**Step 5: Feature Engineering (Outlet Age)**
* **Action**: Create a new feature `Outlet_Years` by subtracting `Outlet_Establishment_Year` from the reference year **2013**.

* **Reasoning**: Raw years like '1985' or '2009' do not directly represent a store's maturity to a machine learning model. Converting them into "**Years of Operation**" provides a more meaningful numerical feature that correlates store experience and brand trust with sales performance.

**Step 6: Dropping Irrelevant Features**
* **Action**: Remove `Item_Identifier` and the original `Outlet_Establishment_Year` column.

* **Reasoning**: `Item_Identifier` contains over **1,500 unique IDs**, which introduces high cardinality and noise, leading to overfitting. `Outlet_Establishment_Year` is now redundant because its predictive information has been fully captured in the new `Outlet_Years` feature.

## 1.1 Standardizing Categorical Labels
Unifying inconsistent entries in the `Item_Fat_Content` column ('LF', 'low fat' $\rightarrow$ 'Low Fat'; 'reg' $\rightarrow$ 'Regular') to ensure clean and uniform categories.

In [439]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


# Load the dataset
file_path = 'data/big_mart_sales.csv' 
df = pd.read_csv(file_path)
df

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.300,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.920,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.500,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.200,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.930,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052
...,...,...,...,...,...,...,...,...,...,...,...,...
8518,FDF22,6.865,Low Fat,0.056783,Snack Foods,214.5218,OUT013,1987,High,Tier 3,Supermarket Type1,2778.3834
8519,FDS36,8.380,Regular,0.046982,Baking Goods,108.1570,OUT045,2002,NaN,Tier 2,Supermarket Type1,549.2850
8520,NCJ29,10.600,Low Fat,0.035186,Health and Hygiene,85.1224,OUT035,2004,Small,Tier 2,Supermarket Type1,1193.1136
8521,FDN46,7.210,Regular,0.145221,Snack Foods,103.1332,OUT018,2009,Medium,Tier 3,Supermarket Type2,1845.5976


In [440]:
print("--- Before Processing: Item_Fat_Content ---")
print(df['Item_Fat_Content'].value_counts())
print("-" * 40)

# Standardize inconsistent categorical labels
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace({
    'LF': 'Low Fat',
    'low fat': 'Low Fat',
    'reg': 'Regular'
})

print("\n--- After Processing: Item_Fat_Content ---")
print(df['Item_Fat_Content'].value_counts())

--- Before Processing: Item_Fat_Content ---
Item_Fat_Content
Low Fat    5089
Regular    2889
LF          316
reg         117
low fat     112
Name: count, dtype: int64
----------------------------------------

--- After Processing: Item_Fat_Content ---
Item_Fat_Content
Low Fat    5517
Regular    3006
Name: count, dtype: int64


## 1.2 Fixing Business Logic Errors
Relabeling non-consumable items (product codes starting with **'NC'**) as **'Non-Edible'** in `Item_Fat_Content`, since household goods cannot logically have a fat content.

In [441]:
# Fix business logic: Items starting with 'NC' are Non-Consumable
df.loc[df['Item_Identifier'].str.startswith('NC'), 'Item_Fat_Content'] = 'Non-Edible'
print("--- After Fixing Logic Errors ---")
print(df['Item_Fat_Content'].value_counts())

--- After Fixing Logic Errors ---
Item_Fat_Content
Low Fat       3918
Regular       3006
Non-Edible    1599
Name: count, dtype: int64


## 1.3 Handling Placeholders (The 0.0 Issue)
Replacing physically impossible `0.0` values in `Item_Visibility` with `NaN`.

In [442]:
# Check the number of 0.0 values before processing
zeros_count = (df['Item_Visibility'] == 0).sum()
print("--- Before Processing: Item_Visibility ---")
print(f"Number of 0 values: {zeros_count}")
print(f"Total NaN values in Item_Visibility: {df['Item_Visibility'].isna().sum()}")
print("-" * 40)

# Replace 0 with NaN
df['Item_Visibility'] = df['Item_Visibility'].replace(0, np.nan)

# Verify the replacement
print("\n--- After Processing: Item_Visibility ---")
print(f"Number of 0 values: {(df['Item_Visibility'] == 0).sum()}")
print(f"Total NaN values in Item_Visibility: {df['Item_Visibility'].isna().sum()}")

--- Before Processing: Item_Visibility ---
Number of 0 values: 526
Total NaN values in Item_Visibility: 0
----------------------------------------

--- After Processing: Item_Visibility ---
Number of 0 values: 0
Total NaN values in Item_Visibility: 526


## 1.4 Imputation for Missing Values
Applying grouped imputation for `Item_Weight` and `Item_Visibility` using the mean of their respective `Item_Identifier`. For `Outlet_Size`, we replace missing values with a new category **'Unknown'** to preserve data integrity and prevent severe bias.

In [443]:
# Check missing values before imputation
print("--- Before Imputation: Missing Values ---")
print(df[['Item_Weight', 'Item_Visibility', 'Outlet_Size']].isnull().sum())
print("-" * 40)

# 1. Impute 'Item_Weight' with the mean weight of the same 'Item_Identifier'
df['Item_Weight'] = df['Item_Weight'].fillna(
    df.groupby('Item_Identifier')['Item_Weight'].transform('mean')
)

# 2. Impute 'Item_Visibility' with the mean visibility of the same 'Item_Identifier'
df['Item_Visibility'] = df['Item_Visibility'].fillna(
    df.groupby('Item_Identifier')['Item_Visibility'].transform('mean')
)

# 3. Impute 'Outlet_Size' with a new category 'Unknown'
df['Outlet_Size'] = df['Outlet_Size'].fillna('Unknown')

# Verify the imputation
print("\n--- After Imputation: Missing Values ---")
print(df[['Item_Weight', 'Item_Visibility', 'Outlet_Size']].isnull().sum())
print("-" * 40)

# Verify the new category in Outlet_Size
print("\nDistribution of Outlet_Size including 'Unknown':")
print(df['Outlet_Size'].value_counts())

--- Before Imputation: Missing Values ---
Item_Weight        1463
Item_Visibility     526
Outlet_Size        2410
dtype: int64
----------------------------------------

--- After Imputation: Missing Values ---
Item_Weight        4
Item_Visibility    0
Outlet_Size        0
dtype: int64
----------------------------------------

Distribution of Outlet_Size including 'Unknown':
Outlet_Size
Medium     2793
Unknown    2410
Small      2388
High        932
Name: count, dtype: int64


#### Investigating Residual Missing Values
After the primary imputation based on `Item_Identifier`, a few records (4 rows) remain missing for `Item_Weight`. This occurs because these specific products either appear only once in the dataset or lack weight data across all their occurrences.  

In [444]:
# Identify and display the remaining 4 rows with missing Item_Weight
residual_missing = df[df['Item_Weight'].isnull()]

print(f"Number of residual missing rows found: {len(residual_missing)}")
display(residual_missing)

Number of residual missing rows found: 4


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
927,FDN52,NaN,Regular,0.130933,Frozen Foods,86.9198,OUT027,1985,Medium,Tier 3,Supermarket Type3,1569.9564
1922,FDK57,NaN,Low Fat,0.079904,Snack Foods,120.0440,OUT027,1985,Medium,Tier 3,Supermarket Type3,4434.2280
4187,FDE52,NaN,Regular,0.029742,Dairy,88.9514,OUT027,1985,Medium,Tier 3,Supermarket Type3,3453.5046
5022,FDQ60,NaN,Regular,0.191501,Baking Goods,121.2098,OUT019,1985,Small,Tier 1,Grocery Store,120.5098


For the remaining 4 items where `Item_Identifier` did not have a historical weight record, we apply a fallback imputation using the **Mean** weight of their respective **`Item_Type`**. This ensures 100% data completeness without losing valuable sales observations.

In [445]:
# Apply fallback imputation for the 4 stubborn NaN values
df['Item_Weight'] = df['Item_Weight'].fillna(
    df.groupby('Item_Type')['Item_Weight'].transform('mean')
)

# Final verification to ensure 0 missing values
print("--- Final Missing Value Count ---")
print(df[['Item_Weight', 'Item_Visibility', 'Outlet_Size']].isnull().sum())

--- Final Missing Value Count ---
Item_Weight        0
Item_Visibility    0
Outlet_Size        0
dtype: int64


## 1.5 Feature Engineering: Outlet Age
The `Outlet_Establishment_Year` represents the year a store was opened. However, for a predictive model, the **age of the outlet** (years of operation) is a more meaningful feature. We will calculate the outlet's age relative to the year the data was collected (2013).

In [446]:
# Calculate the age of the outlet based on the data collection year (2013)
df['Outlet_Years'] = 2013 - df['Outlet_Establishment_Year']

# Verify the new column
print("--- After Feature Engineering: Outlet_Years ---")
print(df[['Outlet_Establishment_Year', 'Outlet_Years']])
print("-" * 40)

# Check the distribution of years
print(df['Outlet_Years'].describe())

--- After Feature Engineering: Outlet_Years ---
      Outlet_Establishment_Year  Outlet_Years
0                          1999            14
1                          2009             4
2                          1999            14
3                          1998            15
4                          1987            26
...                         ...           ...
8518                       1987            26
8519                       2002            11
8520                       2004             9
8521                       2009             4
8522                       1997            16

[8523 rows x 2 columns]
----------------------------------------
count    8523.000000
mean       15.168133
std         8.371760
min         4.000000
25%         9.000000
50%        14.000000
75%        26.000000
max        28.000000
Name: Outlet_Years, dtype: float64


## 1.6 Dropping Irrelevant Features
To streamline the dataset for machine learning, we remove columns that are either redundant or do not contribute predictive value:
* **`Outlet_Establishment_Year`**: Replaced by the more meaningful `Outlet_Years`.
* **`Item_Identifier`**: A unique ID with high cardinality that does not provide generalizable patterns for the model.

In [447]:
# List of columns to drop
cols_to_drop = ['Item_Identifier', 'Outlet_Establishment_Year']

# Drop the columns
df.drop(columns=cols_to_drop, inplace=True)

# Verify the final structure of the cleaned dataset
print("--- Final Columns After Cleaning ---")
print(df.columns.tolist())
print("-" * 40)
print(f"Current Shape: {df.shape}")
display(df)

--- Final Columns After Cleaning ---
['Item_Weight', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type', 'Item_MRP', 'Outlet_Identifier', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type', 'Item_Outlet_Sales', 'Outlet_Years']
----------------------------------------
Current Shape: (8523, 11)


,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,Outlet_Years
0,9.300,Low Fat,0.016047,Dairy,249.8092,OUT049,Medium,Tier 1,Supermarket Type1,3735.1380,14
1,5.920,Regular,0.019278,Soft Drinks,48.2692,OUT018,Medium,Tier 3,Supermarket Type2,443.4228,4
2,17.500,Low Fat,0.016760,Meat,141.6180,OUT049,Medium,Tier 1,Supermarket Type1,2097.2700,14
3,19.200,Regular,0.022911,Fruits and Vegetables,182.0950,OUT010,Unknown,Tier 3,Grocery Store,732.3800,15
4,8.930,Non-Edible,0.016164,Household,53.8614,OUT013,High,Tier 3,Supermarket Type1,994.7052,26
...,...,...,...,...,...,...,...,...,...,...,...
8518,6.865,Low Fat,0.056783,Snack Foods,214.5218,OUT013,High,Tier 3,Supermarket Type1,2778.3834,26
8519,8.380,Regular,0.046982,Baking Goods,108.1570,OUT045,Unknown,Tier 2,Supermarket Type1,549.2850,11
8520,10.600,Non-Edible,0.035186,Health and Hygiene,85.1224,OUT035,Small,Tier 2,Supermarket Type1,1193.1136,9
8521,7.210,Regular,0.145221,Snack Foods,103.1332,OUT018,Medium,Tier 3,Supermarket Type2,1845.5976,4


# 2. Dataset Splitting (Train-Test Split)
* **Description**: We split the dataset into two independent sets: **Training (80%)** and **Testing (20%)** using `train_test_split`.

* **Reasoning**: Splitting the data before any transformations (like Scaling or Log) is the best practice to prevent **Data Leakage**. This ensures that the test set remains completely unseen, allowing us to evaluate the model's performance on truly "new" data.

In [448]:
# Define Features (X) and Target (y)
# We use the raw cleaned data before scaling/log transformation
X = df.drop(columns=['Item_Outlet_Sales'])
y = df['Item_Outlet_Sales']

# Split into Train (80%) and Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Verify the split
print(f"Dataset successfully split:")
print(f"- Training samples: {X_train.shape[0]}")
print(f"- Testing samples:  {X_test.shape[0]}")

Dataset successfully split:
- Training samples: 6818
- Testing samples:  1705


# 3. Data Transformation

**Overview & Workflow Sketch**


**Step 1: Manual Ordinal Mapping**
* **Action**: Manually map `Outlet_Size` and `Outlet_Location_Type` to numerical integers (e.g., Unknown: 0, Small: 1, Medium: 2, High: 3).
* **Reasoning**: Manual mapping preserves the natural hierarchy (Small < Medium < High) and treats '**Unknown**' as the baseline (0), which is more logically consistent than alphabetical Label Encoding.

**Step 2: One-Hot Encoding (N-1 Rule)**
* **Action**: Apply **One-Hot Encoding** to nominal variables (`Outlet_Type`, `Item_Fat_Content`, `Item_Type`, `Outlet_Identifier`)
* **Reasoning**: To avoid the **Dummy Variable Trap**, we transform $N$ categories into $N-1$ binary features. This prevents multicollinearity and ensures mathematical stability for regression models.

**Step 3: Feature Scaling**
* **Action**: Standardize numerical features using StandardScaler.
* **Reasoning**: Features have different scales (Visibility: 0.0–0.3 vs. MRP: 30–260). Scaling them to a mean of 0 and standard deviation of 1 prevents features with larger magnitudes from dominating the model's learning process.

**Step 4: Target Transformation (Log Transformation)**
* **Action**: Apply a natural log transformation (`np.log1p`) to the target variable `Item_Outlet_Sales`.
* **Reasoning**: Sales data is typically **right-skewed**. The log transformation "normalizes" the target distribution, which improves the performance, error variance, and stability of the regression algorithm.

## 3.1 Manual Ordial Mapping
Transform `Outlet_Size` and `Outlet_Location_Type` into numerical ranks to preserve their inherent logical hierarchy.

In [449]:
print("--- Before Manual Ordinal Mapping (X_train) ---")
display(X_train[['Outlet_Size', 'Outlet_Location_Type']].head())
print("-" * 40)

# Define the logical hierarchy for Ordinal Variables
# 'Unknown' is assigned 0 as the baseline
size_map = {'Unknown': 0, 'Small': 1, 'Medium': 2, 'High': 3}
location_map = {'Tier 1': 1, 'Tier 2': 2, 'Tier 3': 3}

# Apply the mapping to both Training and Testing sets
for dataset in [X_train, X_test]:
    dataset['Outlet_Size'] = dataset['Outlet_Size'].map(size_map)
    dataset['Outlet_Location_Type'] = dataset['Outlet_Location_Type'].map(location_map)

# Quick check to verify the mapping in X_train
print("--- After Manual Ordinal Mapping (X_train) ---")
display(X_train[['Outlet_Size', 'Outlet_Location_Type']].head())

--- Before Manual Ordinal Mapping (X_train) ---


,Outlet_Size,Outlet_Location_Type
549,Medium,Tier 1
7757,Unknown,Tier 2
764,Small,Tier 1
6867,Unknown,Tier 2
2716,Small,Tier 1


----------------------------------------
--- After Manual Ordinal Mapping (X_train) ---


,Outlet_Size,Outlet_Location_Type
549,2,1
7757,0,2
764,1,1
6867,0,2
2716,1,1


## 3.2 One-Hot Encoding (N-1 Rule)
This step targets the nominal columns `Outlet_Type`, `Item_Fat_Content`, `Item_Type`, and `Outlet_Identifier`. It converts each unique category within these columns into separate binary features ($0$ or $1$). By applying the $N-1$ rule, we remove one redundant column per category to eliminate multicollinearity, ensuring the model's mathematical stability.

In [450]:
# Define nominal columns
nominal_cols = ['Outlet_Type', 'Item_Fat_Content', 'Item_Type', 'Outlet_Identifier']

# Initialize Encoder (N-1 rule & handle unknown categories)
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# Fit and Transform Training set
encoded_train = encoder.fit_transform(X_train[nominal_cols])
encoded_train_df = pd.DataFrame(encoded_train, 
                                columns=encoder.get_feature_names_out(nominal_cols), 
                                index=X_train.index)

# Transform Testing set ONLY (to avoid data leakage)
encoded_test = encoder.transform(X_test[nominal_cols])
encoded_test_df = pd.DataFrame(encoded_test, 
                               columns=encoder.get_feature_names_out(nominal_cols), 
                               index=X_test.index)

# Drop old columns and merge encoded ones
X_train = pd.concat([X_train.drop(columns=nominal_cols), encoded_train_df], axis=1)
X_test = pd.concat([X_test.drop(columns=nominal_cols), encoded_test_df], axis=1)

# Verification
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
display(X_train)

X_train: (6818, 35) | X_test: (1705, 35)


,Item_Weight,Item_Visibility,Item_MRP,Outlet_Size,Outlet_Location_Type,Outlet_Years,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3,Item_Fat_Content_Non-Edible,...,Item_Type_Starchy Foods,Outlet_Identifier_OUT013,Outlet_Identifier_OUT017,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049
549,9.500,0.035206,171.3448,2,1,14,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
7757,18.000,0.047473,170.5422,0,2,11,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
764,17.600,0.076122,111.7202,1,1,16,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
6867,8.325,0.029845,41.6138,0,2,11,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2716,12.850,0.137228,155.5630,1,1,16,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5734,9.395,0.286345,139.1838,0,3,15,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5191,15.600,0.117575,75.6670,0,2,6,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5390,17.600,0.018944,237.3590,0,2,11,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
860,20.350,0.054363,117.9466,0,2,6,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3.3 Feature Scaling
This code applies `StandardScaler` to both numerical features (`Item_MRP`, `Item_Weight`, `Item_Visibility`, `Outlet_Years`) and mapped ordinal features (`Outlet_Size`, `Outlet_Location_Type`).

In [451]:
# Initialize the StandardScaler
scaler = StandardScaler()

# List of numerical and ordinal columns to be scaled
num_cols = ['Item_MRP', 'Item_Weight', 'Item_Visibility', 'Outlet_Years', 
            'Outlet_Size', 'Outlet_Location_Type']

# FIT & TRANSFORM the Training set
# This learns the mean and std from X_train only
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# 4. TRANSFORM ONLY the Testing set
# This uses the parameters learned from X_train
X_test[num_cols] = scaler.transform(X_test[num_cols])

# Verification
print("--- After Feature Scaling (Post-Split) ---")
display(X_train[num_cols])

--- After Feature Scaling (Post-Split) ---


,Item_MRP,Item_Weight,Item_Visibility,Outlet_Years,Outlet_Size,Outlet_Location_Type
549,0.470709,-0.733786,-0.709733,-0.136169,0.745846,-1.383482
7757,0.457877,1.096432,-0.465070,-0.493521,-1.275290,-0.149659
764,-0.482625,1.010304,0.106311,0.102066,-0.264722,-1.383482
6867,-1.603553,-0.986786,-0.816648,-0.493521,-1.275290,-0.149659
2716,0.218375,-0.012465,1.325033,0.102066,-0.264722,-1.383482
...,...,...,...,...,...,...
5734,-0.043511,-0.756394,4.299081,-0.017052,-1.275290,1.084165
5191,-1.059078,0.579665,0.933060,-1.089109,-1.275290,-0.149659
5390,1.526207,1.010304,-1.034073,-0.493521,-1.275290,-0.149659
860,-0.383072,1.602433,-0.327662,-1.089109,-1.275290,-0.149659


## 3.4 Target Transformation (Log Transformation)
Apply the `np.log1p` ($\log(1+x)$) function to the target variable in both `y_train` and `y_test`.

In [452]:
# Apply log1p transformation (log(1+x)) to the target variable
y_train = np.log1p(y_train)
y_test = np.log1p(y_test)

# Verify transformation
print("--- Target Transformation ---")
print(f"y_train Skewness: {y_train.skew():.2f}")
print(f"y_test Skewness:  {y_test.skew():.2f}")

# Quick look at the transformed target
display(y_train.head())

--- Target Transformation ---
y_train Skewness: -0.89
y_test Skewness:  -0.87


549     7.777888
7757    8.040756
764     7.026606
6867    5.653529
2716    8.348893
Name: Item_Outlet_Sales, dtype: float64

# 4. Saving Processed Data

In [453]:
# 1. Create directories if they don't exist
os.makedirs('train_data', exist_ok=True)
os.makedirs('test_data', exist_ok=True)

# 2. Combine X and y for both sets
train_processed = pd.concat([X_train, y_train], axis=1)
test_processed = pd.concat([X_test, y_test], axis=1)

# 3. Save to CSV without the index column
train_processed.to_csv('train_data/train_processed.csv', index=False)
test_processed.to_csv('test_data/test_processed.csv', index=False)

print("Processed data successfully saved to 'train_data' and 'test_data'.")

Processed data successfully saved to 'train_data' and 'test_data'.
